# Supabase (Postgres)

>[Supabase](https://supabase.com/docs) 是一个开源的 Firebase 替代品。`Supabase` 构建在 `PostgreSQL` 之上，后者提供了强大的 SQL 查询能力，并能够与现有的工具和框架实现简单的集成。

>[PostgreSQL](https://en.wikipedia.org/wiki/PostgreSQL) 也被称为 `Postgres`，是一个免费且开源的关系型数据库管理系统 (RDBMS)，其特点是可扩展性和对 SQL 的兼容性。

本笔记本展示了如何使用 `Supabase` 和 `pgvector` 作为您的 VectorStore。

您需要安装 `langchain-community` (`pip install -qU langchain-community`) 才能使用此集成。

要运行此笔记本，请确保：
- `pgvector` 扩展已启用
- 您已安装 `supabase-py` 包
- 您已在数据库中创建了 `match_documents` 函数
- 您的 `public` schema 中有一个 `documents` 表，其结构与下方类似。

以下函数用于确定余弦相似度，您可以根据需要进行调整。

```sql
-- 启用 pgvector 扩展以使用嵌入向量
create extension if not exists vector;

-- 创建一个表来存储您的文档
create table
  documents (
    id uuid primary key,
    content text, -- 对应 Document.pageContent
    metadata jsonb, -- 对应 Document.metadata
    embedding vector (1536) -- 1536 适用于 OpenAI 嵌入，如有需要请更改
  );

-- 创建一个函数来搜索文档
create function match_documents (
  query_embedding vector (1536),
  filter jsonb default '{}'
) returns table (
  id uuid,
  content text,
  metadata jsonb,
  similarity float
) language plpgsql as $$
#variable_conflict use_column
begin
  return query
  select
    id,
    content,
    metadata,
    1 - (documents.embedding <=> query_embedding) as similarity
  from documents
  where metadata @> filter
  order by documents.embedding <=> query_embedding;
end;
$$;

In [ ]:
# with pip
%pip install --upgrade --quiet  supabase

# with conda
# !conda install -c conda-forge supabase

我们想使用 `OpenAIEmbeddings`，所以需要获取 OpenAI API 密钥。

In [15]:
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [16]:
if "SUPABASE_URL" not in os.environ:
    os.environ["SUPABASE_URL"] = getpass.getpass("Supabase URL:")

In [17]:
if "SUPABASE_SERVICE_KEY" not in os.environ:
    os.environ["SUPABASE_SERVICE_KEY"] = getpass.getpass("Supabase Service Key:")

In [ ]:
# If you're storing your Supabase and OpenAI API keys in a .env file, you can load them with dotenv
from dotenv import load_dotenv

load_dotenv()

首先，我们将创建一个 Supabase 客户端并实例化一个 OpenAI embeddings 类。

In [19]:
import os

from langchain_community.vectorstores import SupabaseVectorStore
from langchain_openai import OpenAIEmbeddings
from supabase.client import Client, create_client

supabase_url = os.environ.get("SUPABASE_URL")
supabase_key = os.environ.get("SUPABASE_SERVICE_KEY")
supabase: Client = create_client(supabase_url, supabase_key)

embeddings = OpenAIEmbeddings()

接下来，我们将加载并解析一些数据以供向量存储使用（如果您已有存储在数据库中的带嵌入的文档，则可以跳过此步骤）。

In [20]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("../../how_to/state_of_the_union.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(documents)

将上述文档插入数据库。系统将自动为每个文档生成嵌入。您可以根据拥有的文档数量调整 `chunk_size`。默认值为 500，但可能需要降低此值。

In [6]:
vector_store = SupabaseVectorStore.from_documents(
    docs,
    embeddings,
    client=supabase,
    table_name="documents",
    query_name="match_documents",
    chunk_size=500,
)

或者，如果您已经在数据库中拥有包含嵌入的文档，请直接实例化一个新的 `SupabaseVectorStore`：

In [10]:
vector_store = SupabaseVectorStore(
    embedding=embeddings,
    client=supabase,
    table_name="documents",
    query_name="match_documents",
)

最后，通过执行相似性搜索来测试它：

In [ ]:
query = "What did the president say about Ketanji Brown Jackson"
matched_docs = vector_store.similarity_search(query)

In [ ]:
print(matched_docs[0].page_content)

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.


## 相似度搜索与评分

Similarity search is a common task in many applications, such as recommendation systems, image retrieval, and natural language processing. It involves finding items in a dataset that are most similar to a given query item.

相似度搜索在许多应用中是一项常见任务，例如推荐系统、图像检索和自然语言处理。它涉及在数据集中查找与给定查询项最相似的项。

The similarity between items is typically measured using a distance metric or a similarity function. Some common metrics include Euclidean distance, cosine similarity, and Jaccard similarity.

项之间的相似度通常使用距离度量或相似度函数来衡量。一些常见的度量包括欧几里得距离、余弦相似度以及 Jaccard 相似度。

Once the similarity is calculated, the results are usually ranked in descending order of similarity score, with the most similar items appearing at the top.

一旦计算出相似度，结果通常会按相似度得分的降序排列，最相似的项会出现在顶部。

### Scoring

Scoring is the process of assigning a numerical value to represent the degree of similarity between two items. The higher the score, the more similar the items are.

### 评分

评分是指为表示两个项之间的相似程度而分配数值的过程。得分越高，项的相似度越高。

Different similarity metrics produce scores on different scales. For example, cosine similarity ranges from -1 to 1, where 1 indicates perfect similarity and -1 indicates perfect dissimilarity. Euclidean distance, on the other hand, is a measure of distance, so lower values indicate higher similarity.

不同的相似度度量会产生不同范围的得分。例如，余弦相似度的范围是 -1 到 1，其中 1 表示完全相似，-1 表示完全不相似。另一方面，欧几里得距离是距离的度量，因此较低的值表示较高的相似度。

When performing similarity search, it's important to choose a similarity metric that is appropriate for the data and the task. The choice of metric can significantly impact the quality of the search results.

在执行相似度搜索时，选择适合数据和任务的相似度度量非常重要。度量的选择会显著影响搜索结果的质量。

### Ranking

Ranking is the process of ordering the search results based on their similarity scores. The items are typically ranked from most similar to least similar.

### 排名

排名是指根据相似度得分对搜索结果进行排序的过程。项通常从最相似到最不相似进行排名。

The ranking can be done in ascending or descending order, depending on the similarity metric used. For metrics where higher values indicate greater similarity (e.g., cosine similarity), the results are ranked in descending order. For metrics where lower values indicate greater similarity (e.g., Euclidean distance), the results are ranked in ascending order.

排名可以按升序或降序进行，具体取决于所使用的相似度度量。对于值越高表示相似度越大的度量（例如余弦相似度），结果按降序排列。对于值越低表示相似度越大的度量（例如欧几里得距离），结果按升序排列。

### Example

Let's consider a simple example of similarity search using cosine similarity. Suppose we have three documents and we want to find the document most similar to a query document.

### 示例

让我们以余弦相似度为例，来看一个简单的相似度搜索示例。假设我们有三个文档，并且我们想找到与查询文档最相似的文档。

**Query Document:** "The quick brown fox jumps over the lazy dog."
**Document 1:** "The quick brown fox is fast."
**Document 2:** "The lazy dog sleeps all day."
**Document 3:** "A quick brown dog jumps over a lazy fox."

**查询文档：** "敏捷的棕色狐狸跳过懒狗。"
**文档 1：** "敏捷的棕色狐狸很快。"
**文档 2：** "懒狗整天睡觉。"
**文档 3：** "一只敏捷的棕色狗跳过一只懒惰的狐狸。"

We can represent these documents as vectors using techniques like TF-IDF or word embeddings. Then, we calculate the cosine similarity between the query document and each of the other documents.

我们可以使用 TF-IDF 或词嵌入等技术将这些文档表示为向量。然后，我们计算查询文档与每个其他文档之间的余弦相似度。

Let's assume the calculated cosine similarities are as follows:

假设计算出的余弦相似度如下：

- Query Document vs. Document 1: 0.75
- Query Document vs. Document 2: 0.20
- Query Document vs. Document 3: 0.60

- 查询文档 vs. 文档 1：0.75
- 查询文档 vs. 文档 2：0.20
- 查询文档 vs. 文档 3：0.60

Based on these scores, we can rank the documents in descending order of similarity:

根据这些得分，我们可以按相似度降序对文档进行排名：

1.  **Document 1** (Score: 0.75)
2.  **Document 3** (Score: 0.60)
3.  **Document 2** (Score: 0.20)

1.  **文档 1** (得分：0.75)
2.  **文档 3** (得分：0.60)
3.  **文档 2** (得分：0.20)

This ranking indicates that Document 1 is the most similar to the query document, followed by Document 3, and then Document 2.

此排名表明文档 1 与查询文档最相似，其次是文档 3，然后是文档 2。

返回的距离分数是余弦距离。因此，分数越低越好。

In [9]:
matched_docs = vector_store.similarity_search_with_relevance_scores(query)

In [10]:
matched_docs[0]

(Document(page_content='Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. \n\nTonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. \n\nOne of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. \n\nAnd I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.', metadata={'source': '../../../state_of_the_union.txt'}),
 0.802509746274066)

## 检索器选项

本节将介绍使用 SupabaseVectorStore 作为检索器的不同选项。

### 最大边际相关性搜索

除了在检索器对象中使用相似性搜索外，您还可以使用 `mmr`。

In [11]:
retriever = vector_store.as_retriever(search_type="mmr")

In [12]:
matched_docs = retriever.invoke(query)

In [13]:
for i, d in enumerate(matched_docs):
    print(f"\n## Document {i}\n")
    print(d.page_content)


## Document 0

Tonight. I call on the Senate to: Pass the Freedom to Vote Act. Pass the John Lewis Voting Rights Act. And while you’re at it, pass the Disclose Act so Americans can know who is funding our elections. 

Tonight, I’d like to honor someone who has dedicated his life to serve this country: Justice Stephen Breyer—an Army veteran, Constitutional scholar, and retiring Justice of the United States Supreme Court. Justice Breyer, thank you for your service. 

One of the most serious constitutional responsibilities a President has is nominating someone to serve on the United States Supreme Court. 

And I did that 4 days ago, when I nominated Circuit Court of Appeals Judge Ketanji Brown Jackson. One of our nation’s top legal minds, who will continue Justice Breyer’s legacy of excellence.

## Document 1

One was stationed at bases and breathing in toxic smoke from “burn pits” that incinerated wastes of war—medical and hazard material, jet fuel, and more. 

When they came home, many